In [0]:
%python
# Databricks notebook source
from pyspark.sql import functions as F

CATALOG = "vf_health"
SCHEMA = "ghana"
BRONZE = f"{CATALOG}.{SCHEMA}.bronze_raw_facilities"

INPUT_PATH = "/Volumes/vf_health/ghana/raw/Virtue Foundation Ghana v0.3 - Sheet1.csv"

# Read raw CSV safely
df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(INPUT_PATH)
)

# Fix invalid header names from source CSV
if "mongo DB" in df.columns:
    df = df.withColumnRenamed("mongo DB", "mongo_db")

# Remove fully-empty rows (dataset has many trailing empty rows)
non_empty = None
for c in df.columns:
    cond = F.col(c).isNotNull() & (F.trim(F.col(c)) != "")
    non_empty = cond if non_empty is None else (non_empty | cond)

df = df.filter(non_empty)

# Add ingestion timestamp
df = df.withColumn("ingested_at", F.current_timestamp())

# Write to Bronze
df.write.mode("overwrite").format("delta").saveAsTable(BRONZE)

print("Bronze count:", spark.table(BRONZE).count())

In [0]:
SELECT * FROM vf_health.ghana.bronze_raw_facilities LIMIT 20